# Facebook

**Dr. Dave Wanik - University of Connecticut - Dept. of Operations and Information Management**

---------------------------

In [1]:
!pip install gdown==4.6.0

  Attempting uninstall: gdown
    Found existing installation: gdown 4.6.6
    Uninstalling gdown-4.6.6:
      Successfully uninstalled gdown-4.6.6


In [2]:
# https://docs.google.com/spreadsheets/d/1gg3tTgale5mGdjImqB692gP6U0WUki_6/edit?usp=sharing&ouid=118154520763102479420&rtpof=true&sd=true

!gdown 1gg3tTgale5mGdjImqB692gP6U0WUki_6

Downloading...
From: https://drive.google.com/uc?id=1gg3tTgale5mGdjImqB692gP6U0WUki_6
To: /content/Facebook Survey - Post(2).xlsx
100% 13.2k/13.2k [00:00<00:00, 38.0MB/s]


# Read the Data

## First attempt

In [3]:
import pandas as pd

df = pd.read_csv('/content/Facebook Survey - Post(2).xlsx') # WOMP WOMP

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xdb in position 71: invalid continuation byte

Why does this fail? Because it's an excel file and not a .csv file. Try using `read_excel`.

## Second Attempt

In [4]:
# you can ignore this warning message - xlsx files are always tough in python
# csv files are better!
df = pd.read_excel('/content/Facebook Survey - Post(2).xlsx')

It looks like it read the file, but we need to look to be sure...

In [5]:
df.head()

,Facebook Survey,Unnamed: 1,Unnamed: 2,Unnamed: 3
0,NaN,NaN,NaN,NaN
1,Student,Gender,Hours online/week,Friends
2,1,female,4,150
3,2,female,10,400
4,3,male,7,120


😞 Oh no! It read the file, but it read that title that an analyst wrote in the top of the worksheet (open in Excel or Google Sheets to check.)

## Third attempt

Let's skip the first two rows then try again.

In [6]:
df = pd.read_excel('/content/Facebook Survey - Post(2).xlsx', skiprows=2) # note the skiprows argument
df.head() # check your work

,Student,Gender,Hours online/week,Friends
0,1,female,4,150
1,2,female,10,400
2,3,male,7,120
3,4,male,15,500
4,5,female,9,260


Ta da! Now I was able to read the file.

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Student            33 non-null     int64 
 1   Gender             33 non-null     object
 2   Hours online/week  33 non-null     int64 
 3   Friends            33 non-null     int64 
dtypes: int64(3), object(1)
memory usage: 1.2+ KB


# Question 1: means per group
Based on the data, create a pivot table that displays both the average of hours online/week and average of friends per gender.
* What is the difference in the average number of hours on Facebook for male versus female students?
* What is the difference in the average number of friends on Facebook for male versus female students?

## First things first...

We need to ensure data is numeric if we are going to perform mathematical operations! You can't calculate 'apple'/2....

In [ ]:
df.info() # check all datatypes and missing rows

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Student            33 non-null     int64 
 1   Gender             33 non-null     object
 2   Hours online/week  33 non-null     int64 
 3   Friends            33 non-null     int64 
dtypes: int64(3), object(1)
memory usage: 1.2+ KB


In [ ]:
df.head() # and let's refresh on what the data looks like

,Student,Gender,Hours online/week,Friends
0,1,female,4,150
1,2,female,10,400
2,3,male,7,120
3,4,male,15,500
4,5,female,9,260


It looks like we have a column for the student (an ID column with integer data type), the Gender (stored as an object or 'string' data type), the number of hours online (integer) and the count of friends (integer.) We should have no problem exploring this dataset.

### 🔴 What is the difference in the average number of hours on Facebook for male versus female students?

Of course, you could use Generative AI here to help... but it is good to be able to visualize what this is on your own. You need to calculate the mean per group.

In [ ]:
# prompt: What is the difference in the average number of hours on Facebook for male versus female students?

df.groupby('Gender').agg({'Hours online/week': 'mean'})


,Hours online/week
Gender,
female,6.150000
male,6.384615


There are MANY different ways you could answer this question! Here is another way, but it returns all of the columns.

In [8]:
df.groupby('Gender').mean()

,Student,Hours online/week,Friends
Gender,,,
female,16.300000,6.150000,157.000000
male,18.076923,6.384615,207.692308


Resetting the index is also a good idea.

In [9]:
df.groupby('Gender').mean().reset_index()

,Gender,Student,Hours online/week,Friends
0,female,16.300000,6.150000,157.000000
1,male,18.076923,6.384615,207.692308


And if you wanted these results for later, you could save it for later.

In [10]:
tmp = df.groupby('Gender').mean().reset_index()
# you could even save it as a .csv for later!
tmp.to_csv('myOutput.csv')

### 🔴 What is the difference in the average number of friends on Facebook for male versus female students?

In [15]:
df.groupby('Gender').agg({'Friends': 'mean',
                          'Hours online/week' : 'mean'}) # similar to before

,Friends,Hours online/week
Gender,,
female,157.000000,6.150000
male,207.692308,6.384615


# Question 2: 'High Advertising'

### 🔴 Add a new column with a single (logical) statement that displays "High Advertising" if it meets all of the conditions below and otherwise displays "Low Advertising"

* Gender: male
* Hours online/week greater than 6
* Friends greater than 150

This is an example of using multiple if conditions, which will require the use of () and & for the multiple conditions. Each condition needs its own (). Like this!

In [16]:
import numpy as np

In [17]:
df['Advertising'] = np.where(
                              (df['Gender'] == 'male') &
                              (df['Hours online/week'] > 6) &
                              (df['Friends'] > 150),
                              'HIGH ADVERTISING', # if true
                              'LOW ADVERTISING' # if false (otherwise)
)

😀 It worked!

In [18]:
df.head(n=10) # check your work

,Student,Gender,Hours online/week,Friends,Advertising
0,1,female,4,150,LOW ADVERTISING
1,2,female,10,400,LOW ADVERTISING
2,3,male,7,120,LOW ADVERTISING
3,4,male,15,500,HIGH ADVERTISING
4,5,female,9,260,LOW ADVERTISING
5,6,female,5,70,LOW ADVERTISING
6,7,female,7,90,LOW ADVERTISING
7,8,male,5,250,LOW ADVERTISING
8,9,female,12,110,LOW ADVERTISING
9,10,female,2,30,LOW ADVERTISING


# Question 3

### 🔴Create a single logical statement that shows "Low" if the friends count is below 100, "Medium" if it's below 300, and "High" if it's 300 or greater

This is an example of using nested if/logical statements. There are many different ways to write this query! Do what makes sense to you.

In [19]:
df['FLAG_Friends'] = np.where(df['Friends'] > 300,
                              'HIGH', # if true
                              np.where(df['Friends'] < 100, # if false (first np.where)
                                       'LOW', # if true
                                        'MEDIUM')) # if false

In [20]:
df.head(n=10) # check your work!

,Student,Gender,Hours online/week,Friends,Advertising,FLAG_Friends
0,1,female,4,150,LOW ADVERTISING,MEDIUM
1,2,female,10,400,LOW ADVERTISING,HIGH
2,3,male,7,120,LOW ADVERTISING,MEDIUM
3,4,male,15,500,HIGH ADVERTISING,HIGH
4,5,female,9,260,LOW ADVERTISING,MEDIUM
5,6,female,5,70,LOW ADVERTISING,LOW
6,7,female,7,90,LOW ADVERTISING,LOW
7,8,male,5,250,LOW ADVERTISING,MEDIUM
8,9,female,12,110,LOW ADVERTISING,MEDIUM
9,10,female,2,30,LOW ADVERTISING,LOW


In [ ]:
df['FLAG_Friends'].value_counts() # do you get the same answer with your method?

MEDIUM    14
LOW       13
HIGH       6
Name: FLAG_Friends, dtype: int64

# Question 4

### 🔴 Create a single statement that shows "High End User" if one of the conditions below are met OR "Low End User" if neither are met
* Hours online/week greater than 8
* Friends greater than 300

In [21]:
df['FLAG_user'] = np.where(
                            (df['Hours online/week'] > 8) | (df['Friends'] > 300),
                            'HIGH END USER', # if true
                            'LOW END USER') # if false

In [ ]:
df.head(n=10) # check your work!

,Student,Gender,Hours online/week,Friends,Advertising,FLAG_Friends,FLAG_user
0,1,female,4,150,LOW ADVERTISING,MEDIUM,LOW END USER
1,2,female,10,400,LOW ADVERTISING,HIGH,HIGH END USER
2,3,male,7,120,LOW ADVERTISING,MEDIUM,LOW END USER
3,4,male,15,500,HIGH ADVERTISING,HIGH,HIGH END USER
4,5,female,9,260,LOW ADVERTISING,MEDIUM,LOW END USER
5,6,female,5,70,LOW ADVERTISING,LOW,LOW END USER
6,7,female,7,90,LOW ADVERTISING,LOW,LOW END USER
7,8,male,5,250,LOW ADVERTISING,MEDIUM,LOW END USER
8,9,female,12,110,LOW ADVERTISING,MEDIUM,LOW END USER
9,10,female,2,30,LOW ADVERTISING,LOW,LOW END USER


In [ ]:
df['FLAG_user'].value_counts()

LOW END USER     29
HIGH END USER     4
Name: FLAG_user, dtype: int64